In [3]:
import os, random
import numpy as np
import pandas as pd
from datetime import date, datetime, timedelta

# --- settings ---
SEED = 42
N_CLAIMS = 50_000
N_PROVIDERS = 350
OUT_DIR = "output_rcm"

random.seed(SEED)
np.random.seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)

LOCATIONS = [
    "Providence, RI", "Pawtucket, RI", "Warwick, RI", "Attleboro, MA", "Boston, MA",
    "New Bedford, MA", "Worcester, MA", "Springfield, MA"
]
SPECIALTIES = [
    "Primary Care", "Cardiology", "Orthopedics", "Emergency", "Radiology",
    "Behavioral Health", "Endocrinology", "Gastroenterology", "OB/GYN", "Neurology"
]
PAYERS_CANONICAL = [
    "Blue Cross Blue Shield", "Aetna", "UnitedHealthcare", "Cigna",
    "Humana", "Medicare", "Medicaid", "Self-Pay", "Tricare"
]
PAYER_PLAN_MAP = {
    "Blue Cross Blue Shield": "Commercial",
    "Aetna": "Commercial",
    "UnitedHealthcare": "Commercial",
    "Cigna": "Commercial",
    "Humana": "Commercial",
    "Medicare": "Medicare",
    "Medicaid": "Medicaid",
    "Self-Pay": "Self-Pay",
    "Tricare": "Military",
}
PAYER_MIX_WEIGHTS = np.array([0.17, 0.09, 0.16, 0.08, 0.06, 0.20, 0.18, 0.04, 0.02])
PAYER_MIX_WEIGHTS = PAYER_MIX_WEIGHTS / PAYER_MIX_WEIGHTS.sum()

DENIAL_CATEGORIES = ["Eligibility", "Authorization", "Coding", "Medical Necessity", "Timely Filing", "Other"]
DENIAL_REASONS_BY_CAT = {
    "Eligibility": ["Coverage inactive", "Member not found", "Invalid policy number", "COB required", "Eligibility not verified"],
    "Authorization": ["No prior authorization", "Auth expired", "Incorrect auth number", "Auth not on file"],
    "Coding": ["Invalid CPT", "Modifier missing", "ICD mismatch", "Duplicate claim", "Bundling issue", "NPI mismatch"],
    "Medical Necessity": ["Not medically necessary", "Documentation insufficient", "Frequency exceeded", "Experimental/investigational"],
    "Timely Filing": ["Filing limit exceeded", "Late submission", "Timely filing documentation missing"],
    "Other": ["Coordination of benefits", "Provider not credentialed", "Missing attachments", "Claim incomplete"],
}
BASE_DENIAL_RATE_BY_PLAN = {"Commercial": 0.14, "Medicare": 0.10, "Medicaid": 0.18, "Self-Pay": 0.05, "Military": 0.12}
PAY_DELAY_MEAN_BY_PLAN = {"Commercial": 28, "Medicare": 21, "Medicaid": 35, "Self-Pay": 15, "Military": 30}

START_DATE = date(2023, 1, 1)
END_DATE = date(2025, 12, 31)

def random_date(start: date, end: date) -> date:
    delta_days = (end - start).days
    return start + timedelta(days=int(np.random.randint(0, delta_days + 1)))

def to_mixed_date_format(d: date) -> str:
    fmts = ["%Y-%m-%d", "%m/%d/%Y", "%d-%b-%Y"]
    fmt = np.random.choice(fmts, p=[0.75, 0.20, 0.05])
    return d.strftime(fmt)

def messy_payer_name(canonical: str) -> str:
    variants = {
        "Blue Cross Blue Shield": ["BCBS", "BlueCross", "Blue Cross", "Blue Cross Blue Shield", "BlueCross BlueShield"],
        "Aetna": ["Aetna", "AETNA", "Aetna Health", "Aetna Inc."],
        "UnitedHealthcare": ["UnitedHealthcare", "UHC", "United Health", "United HealthCare"],
        "Cigna": ["Cigna", "CIGNA", "Cigna Health"],
        "Humana": ["Humana", "HUMANA", "Humana Inc."],
        "Medicare": ["Medicare", "MEDICARE", "Traditional Medicare"],
        "Medicaid": ["Medicaid", "MEDICAID", "State Medicaid"],
        "Self-Pay": ["Self-Pay", "Self Pay", "SELFPAY", "Cash Pay"],
        "Tricare": ["Tricare", "TRICARE", "TriCare"],
    }
    return str(np.random.choice(variants.get(canonical, [canonical])))

print("Config:", {"N_CLAIMS": N_CLAIMS, "N_PROVIDERS": N_PROVIDERS, "OUT_DIR": OUT_DIR})

Config: {'N_CLAIMS': 50000, 'N_PROVIDERS': 350, 'OUT_DIR': 'output_rcm'}


In [10]:
# --- Providers ---
rng = np.random.default_rng(SEED)

provider_ids = [f"PROV{str(i).zfill(5)}" for i in range(1, N_PROVIDERS + 1)]

providers = pd.DataFrame({
    "ProviderID": provider_ids,
    "NPI": rng.integers(1_000_000_000, 10_000_000_000, size=N_PROVIDERS, dtype=np.int64).astype(str),
    "Specialty": rng.choice(SPECIALTIES, size=N_PROVIDERS),
    "Location": rng.choice(LOCATIONS, size=N_PROVIDERS),
})

# tiny mess
providers.loc[providers.sample(frac=0.02, random_state=SEED).index, "Specialty"] = None
providers.loc[providers.sample(frac=0.01, random_state=SEED + 1).index, "Location"] = None

# --- Payers ---
payers = pd.DataFrame({
    "PayerCanonical": PAYERS_CANONICAL,
    "PlanType": [PAYER_PLAN_MAP[p] for p in PAYERS_CANONICAL],
})

# --- Claims ---
claim_ids = [f"CLM{str(i).zfill(8)}" for i in range(1, N_CLAIMS + 1)]
patient_ids = [f"PAT{str(np.random.randint(1, N_CLAIMS // 2 + 1)).zfill(7)}" for _ in claim_ids]

payer_canon = np.random.choice(PAYERS_CANONICAL, size=N_CLAIMS, p=PAYER_MIX_WEIGHTS)
payer_messy = [messy_payer_name(p) for p in payer_canon]
plan_types = [PAYER_PLAN_MAP[p] for p in payer_canon]

service_dates = [random_date(START_DATE, END_DATE) for _ in range(N_CLAIMS)]
submit_dates = []
for sd, pt in zip(service_dates, plan_types):
    lag = int(np.clip(np.random.normal(loc=4, scale=3), 0, 30))
    if pt == "Medicaid" and np.random.rand() < 0.15:
        lag += int(np.random.randint(5, 20))
    submit_dates.append(sd + timedelta(days=lag))

billed = np.round(np.random.gamma(shape=2.0, scale=250.0, size=N_CLAIMS) + 50, 2)
allowed = np.round(billed * np.random.uniform(0.35, 0.85, size=N_CLAIMS), 2)

statuses = []
for pt in plan_types:
    base = BASE_DENIAL_RATE_BY_PLAN[pt]
    r = np.random.rand()
    if r < base:
        statuses.append("Denied")
    elif r < base + 0.72:
        statuses.append("Paid")
    elif r < base + 0.90:
        statuses.append("Partial")
    else:
        statuses.append("Submitted")

claims = pd.DataFrame({
    "ClaimID": claim_ids,
    "PatientID": patient_ids,
    "ProviderID": np.random.choice(provider_ids, size=N_CLAIMS),
    "Location": np.random.choice(LOCATIONS, size=N_CLAIMS),
    "Specialty": np.random.choice(SPECIALTIES, size=N_CLAIMS),
    "ServiceDate": [to_mixed_date_format(d) for d in service_dates],
    "SubmitDate": [to_mixed_date_format(d) for d in submit_dates],
    "Payer": payer_messy,
    "PayerCanonical": payer_canon,
    "PlanType": plan_types,
    "BilledAmount": billed,
    "AllowedAmount": allowed,
    "ClaimStatus": statuses,
})

# mess: missing allowed amounts
claims.loc[claims.sample(frac=0.015, random_state=SEED).index, "AllowedAmount"] = np.nan
# mess: inconsistent status casing
ix = claims.sample(frac=0.01, random_state=SEED + 8).index
claims.loc[ix, "ClaimStatus"] = claims.loc[ix, "ClaimStatus"].str.upper()
# mess: whitespace noise in payer strings
ix2 = claims.sample(frac=0.02, random_state=SEED + 9).index
claims.loc[ix2, "Payer"] = claims.loc[ix2, "Payer"].astype(str).apply(lambda x: f" {x} " if np.random.rand() < 0.5 else x)
# mess: duplicate some rows (duplicate ClaimIDs)
dup_rows = claims.sample(frac=0.008, random_state=SEED + 7)
claims = pd.concat([claims, dup_rows], ignore_index=True)

# --- Denials ---
unique_claims = claims.drop_duplicates(subset=["ClaimID"]).copy()
denial_rows = []
for _, row in unique_claims.iterrows():
    plan = row["PlanType"]
    base = BASE_DENIAL_RATE_BY_PLAN.get(plan, 0.12)
    payer_bump = 0.03 if row["PayerCanonical"] == "Medicaid" else 0.0
    p_deny = float(np.clip(base + payer_bump + np.random.normal(0, 0.02), 0.01, 0.40))

    r = np.random.rand()
    n_events = 0
    if r < p_deny:
        n_events = 1
        if np.random.rand() < 0.18:
            n_events = 2

    if n_events == 0:
        continue

    sd = pd.to_datetime(row["ServiceDate"], errors="coerce")
    sub = pd.to_datetime(row["SubmitDate"], errors="coerce")
    anchor = sub if pd.notna(sub) else sd
    if pd.isna(anchor):
        anchor = pd.Timestamp(datetime(2024, 1, 1))

    for e in range(n_events):
        cat = str(np.random.choice(DENIAL_CATEGORIES, p=[0.22, 0.16, 0.28, 0.16, 0.10, 0.08]))
        reason = str(np.random.choice(DENIAL_REASONS_BY_CAT[cat]))
        denial_lag = int(np.clip(np.random.normal(loc=10 + e * 7, scale=6), 1, 60))
        denial_date = (anchor + pd.Timedelta(days=denial_lag)).date()

        appealed = bool(np.random.rand() < (0.35 if cat in ["Medical Necessity", "Authorization"] else 0.18))
        resub = bool(np.random.rand() < (0.55 if cat in ["Coding", "Eligibility"] else 0.30))

        if appealed:
            outcome = str(np.random.choice(["Overturned", "Upheld", "Pending"], p=[0.35, 0.25, 0.40]))
        else:
            outcome = str(np.random.choice(["Not Appealed", "Pending"], p=[0.75, 0.25]))

        denial_rows.append({
            "DenialID": f"DNY{row['ClaimID']}_{e+1}",
            "ClaimID": row["ClaimID"],
            "DenialDate": to_mixed_date_format(denial_date),
            "DenialCategory": cat,
            "DenialReason": reason,
            "AppealedFlag": appealed,
            "ResubmittedFlag": resub,
            "Outcome": outcome,
        })

denials = pd.DataFrame(denial_rows)
if not denials.empty:
    # mess: inconsistent category labels
    ix3 = denials.sample(frac=0.01, random_state=SEED + 20).index
    denials.loc[ix3, "DenialCategory"] = denials.loc[ix3, "DenialCategory"].replace({
        "Medical Necessity": "Med Necessity",
        "Authorization": "Auth",
        "Timely Filing": "Timely",
    })
    # mess: missing denial reason
    denials.loc[denials.sample(frac=0.007, random_state=SEED + 21).index, "DenialReason"] = None

# --- Payments ---
denial_claims = set(denials["ClaimID"].unique()) if not denials.empty else set()
pay_rows = []

for _, row in unique_claims.iterrows():
    claim_id = row["ClaimID"]
    status = str(row["ClaimStatus"]).strip()
    plan = row["PlanType"]
    allowed_amt = row["AllowedAmount"]

    # skip some unpaid
    if status.upper() == "SUBMITTED" and np.random.rand() < 0.75:
        continue
    if claim_id in denial_claims and np.random.rand() < 0.45:
        continue

    n_pay = 1
    if status.upper() == "PARTIAL" and np.random.rand() < 0.55:
        n_pay = int(np.random.choice([2, 3], p=[0.75, 0.25]))
    elif np.random.rand() < 0.08:
        n_pay = 2

    sd = pd.to_datetime(row["ServiceDate"], errors="coerce")
    if pd.isna(sd):
        sd = pd.Timestamp(datetime(2024, 1, 1))

    mean_delay = PAY_DELAY_MEAN_BY_PLAN.get(plan, 28)
    if claim_id in denial_claims:
        mean_delay += 18

    if pd.isna(allowed_amt) or allowed_amt is None:
        base_amt = float(row["BilledAmount"]) * float(np.random.uniform(0.35, 0.75))
    else:
        base_amt = float(allowed_amt)

    if np.random.rand() < 0.02:
        base_amt = 0.0

    splits = np.random.dirichlet(np.ones(n_pay), size=1)[0]
    amounts = np.round(base_amt * splits, 2)

    for i in range(n_pay):
        delay = int(np.clip(np.random.normal(loc=mean_delay + (i * 7), scale=10), 3, 180))
        pay_date = (sd + pd.Timedelta(days=delay)).date()
        pay_rows.append({
            "PaymentID": f"PAY{claim_id}_{i+1}",
            "ClaimID": claim_id,
            "PaymentDate": to_mixed_date_format(pay_date),
            "PaymentAmount": amounts[i],
        })

payments = pd.DataFrame(pay_rows)
if not payments.empty:
    # mess: currency symbols/commas in amount strings for a subset
    ix4 = payments.sample(frac=0.01, random_state=SEED + 30).index
    payments.loc[ix4, "PaymentAmount"] = payments.loc[ix4, "PaymentAmount"].apply(
        lambda x: f"${x:,.2f}" if np.random.rand() < 0.7 else f"{x:,}"
    )
    # mess: missing payment dates
    payments.loc[payments.sample(frac=0.004, random_state=SEED + 31).index, "PaymentDate"] = None

# --- Export ---
providers.to_csv(os.path.join(OUT_DIR, "Providers.csv"), index=False)
payers.to_csv(os.path.join(OUT_DIR, "Payers.csv"), index=False)
claims.to_csv(os.path.join(OUT_DIR, "Claims.csv"), index=False)
denials.to_csv(os.path.join(OUT_DIR, "Denials.csv"), index=False)
payments.to_csv(os.path.join(OUT_DIR, "Payments.csv"), index=False)

(len(claims), len(denials), len(payments), OUT_DIR)

C:\Users\christopherfontes\AppData\Local\Temp\ipykernel_8704\3009834473.py:200: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['$237.57' '$483.05' '$117.64' '$332.17' '$305.08' '$629.29' '$86.03'
 '$252.61' '$239.44' '436.4' '11.84' '$986.73' '$328.86' '264.74'
 '$220.33' '$90.22' '$0.00' '77.58' '653.92' '$123.36' '160.73' '$580.25'
 '$465.96' '$421.47' '$546.91' '0.0' '$264.15' '$91.25' '$33.00' '$560.73'
 '$279.75' '$710.50' '$624.91' '4.43' '$304.93' '$800.89' '$68.91' '21.01'
 '330.11' '$197.61' '$314.95' '843.49' '324.46' '$153.38' '$198.74'
 '$576.37' '380.32' '423.0' '$3.45' '98.17' '$171.09' '$566.06' '$201.47'
 '$330.33' '$471.93' '$1.73' '69.97' '$83.31' '$57.05' '$368.05' '$551.60'
 '$490.60' '$355.74' '251.78' '$21.44' '$31.09' '$58.43' '$193.40'
 '$246.77' '165.73' '0.0' '490.41' '$457.38' '$124.33' '$38.25' '$97.91'
 '180.93' '$290.38' '$191.91' '$1,518.87' '$250.58' '$112.50' '$583.60'
 '

(50400, 8407, 54739, 'output_rcm')

In [11]:
payments["PaymentAmount"] = payments["PaymentAmount"].astype(object)

In [12]:
payments["PaymentAmountRaw"] = payments["PaymentAmount"]

ix4 = payments.sample(frac=0.01, random_state=SEED + 30).index
payments.loc[ix4, "PaymentAmountRaw"] = payments.loc[ix4, "PaymentAmountRaw"].apply(
    lambda x: f"${x:,.2f}" if np.random.rand() < 0.7 else f"{x:,}"
)

ValueError: Unknown format code 'f' for object of type 'str'

In [13]:
def to_messy_money(x):
    if pd.isna(x):
        return x
    # if it's already a string, just return it (or strip)
    if isinstance(x, str):
        return x.strip()
    # otherwise treat as number
    x = float(x)
    return f"${x:,.2f}" if np.random.rand() < 0.7 else f"{x:,.2f}".replace(".00", "")

In [14]:
payments["PaymentAmount"] = pd.to_numeric(payments["PaymentAmount"], errors="coerce")
payments["PaymentAmountRaw"] = payments["PaymentAmount"]

In [15]:
providers.head()

,ProviderID,NPI,Specialty,Location
0,PROV00001,7965604437,Behavioral Health,"Boston, MA"
1,PROV00002,4949905957,Endocrinology,"Warwick, RI"
2,PROV00003,8727381279,Cardiology,"Boston, MA"
3,PROV00004,7276312261,Cardiology,"Springfield, MA"
4,PROV00005,1847596130,Cardiology,"Providence, RI"


In [16]:
claims.head()

,ClaimID,PatientID,ProviderID,Location,Specialty,ServiceDate,SubmitDate,Payer,PayerCanonical,PlanType,BilledAmount,AllowedAmount,ClaimStatus
0,CLM00000001,PAT0023655,PROV00349,"Pawtucket, RI",Emergency,2024-01-15,2024-01-22,Traditional Medicare,Medicare,Medicare,663.41,267.55,Paid
1,CLM00000002,PAT0015796,PROV00034,"Warwick, RI",Emergency,2023-12-12,2023-12-20,MEDICAID,Medicaid,Medicaid,168.58,109.33,Paid
2,CLM00000003,PAT0000861,PROV00112,"Providence, RI",Emergency,2023-01-15,2023-01-19,TriCare,Tricare,Military,469.26,246.67,Denied
3,CLM00000004,PAT0005391,PROV00085,"New Bedford, MA",Primary Care,2025-11-08,2025-11-10,United HealthCare,UnitedHealthcare,Commercial,183.47,155.86,Paid
4,CLM00000005,PAT0021576,PROV00271,"Providence, RI",Behavioral Health,2025-09-22,09/24/2025,Humana,Humana,Commercial,387.40,NaN,Paid


In [17]:
denials.head()

,DenialID,ClaimID,DenialDate,DenialCategory,DenialReason,AppealedFlag,ResubmittedFlag,Outcome
0,DNYCLM00000010_1,CLM00000010,2023-07-13,Other,Coordination of benefits,False,True,Not Appealed
1,DNYCLM00000010_2,CLM00000010,07/17/2023,Coding,Duplicate claim,False,True,Pending
2,DNYCLM00000012_1,CLM00000012,2024-08-19,Coding,ICD mismatch,False,True,Not Appealed
3,DNYCLM00000018_1,CLM00000018,2024-11-12,Authorization,Auth expired,False,False,Not Appealed
4,DNYCLM00000028_1,CLM00000028,2024-09-08,Other,Missing attachments,False,False,Not Appealed


In [18]:
payments.head()

,PaymentID,ClaimID,PaymentDate,PaymentAmount,PaymentAmountRaw
0,PAYCLM00000001_1,CLM00000001,2024-02-05,267.55,267.55
1,PAYCLM00000002_1,CLM00000002,01/02/2024,94.36,94.36
2,PAYCLM00000002_2,CLM00000002,01/14/2024,14.97,14.97
3,PAYCLM00000003_1,CLM00000003,01/29/2023,246.67,246.67
4,PAYCLM00000004_1,CLM00000004,2025-12-09,155.86,155.86


In [19]:
claims.sample(10, random_state=SEED)

,ClaimID,PatientID,ProviderID,Location,Specialty,ServiceDate,SubmitDate,Payer,PayerCanonical,PlanType,BilledAmount,AllowedAmount,ClaimStatus
45078,CLM00045079,PAT0020229,PROV00233,"Warwick, RI",OB/GYN,2024-12-27,2025-01-03,State Medicaid,Medicaid,Medicaid,264.29,166.43,Denied
36590,CLM00036591,PAT0021903,PROV00059,"New Bedford, MA",Cardiology,2023-10-07,2023-10-18,Blue Cross Blue Shield,Blue Cross Blue Shield,Commercial,511.10,374.01,Paid
9430,CLM00009431,PAT0021741,PROV00032,"Worcester, MA",OB/GYN,01/13/2025,2025-01-15,MEDICAID,Medicaid,Medicaid,179.54,144.10,Denied
7096,CLM00007097,PAT0002213,PROV00123,"Providence, RI",Neurology,30-Nov-2024,12/07/2024,TRICARE,Tricare,Military,522.86,210.28,Paid
20418,CLM00020419,PAT0011305,PROV00243,"Warwick, RI",OB/GYN,2024-08-08,2024-08-13,MEDICARE,Medicare,Medicare,769.98,420.12,Paid
3767,CLM00003768,PAT0023348,PROV00182,"Pawtucket, RI",Emergency,2025-03-04,2025-03-09,BCBS,Blue Cross Blue Shield,Commercial,553.44,360.39,Paid
14164,CLM00014165,PAT0023872,PROV00330,"Springfield, MA",Gastroenterology,05/07/2024,05/11/2024,Medicare,Medicare,Medicare,1000.71,773.87,Partial
48071,CLM00048072,PAT0015096,PROV00269,"New Bedford, MA",Orthopedics,05/15/2023,2023-05-22,CIGNA,Cigna,Commercial,1082.38,529.85,Paid
20628,CLM00020629,PAT0017955,PROV00337,"Providence, RI",Cardiology,2025-02-09,02/13/2025,AETNA,Aetna,Commercial,1512.90,555.63,Partial
508,CLM00000509,PAT0023389,PROV00061,"Attleboro, MA",Primary Care,2024-04-24,2024-04-28,MEDICARE,Medicare,Medicare,510.06,NaN,Paid


In [20]:
import os

OUT_DIR = "output_rcm"
os.makedirs(OUT_DIR, exist_ok=True)

providers.to_csv(os.path.join(OUT_DIR, "Providers.csv"), index=False)
payers.to_csv(os.path.join(OUT_DIR, "Payers.csv"), index=False)
claims.to_csv(os.path.join(OUT_DIR, "Claims.csv"), index=False)
denials.to_csv(os.path.join(OUT_DIR, "Denials.csv"), index=False)
payments.to_csv(os.path.join(OUT_DIR, "Payments.csv"), index=False)

# confirmation
for f in ["Providers.csv", "Payers.csv", "Claims.csv", "Denials.csv", "Payments.csv"]:
    path = os.path.join(OUT_DIR, f)
    print(f, "->", os.path.getsize(path), "bytes")

Providers.csv -> 17574 bytes
Payers.csv -> 219 bytes
Claims.csv -> 6928491 bytes
Denials.csv -> 791924 bytes
Payments.csv -> 2967755 bytes


In [21]:
import os
os.path.abspath(OUT_DIR)

'C:\\Users\\christopherfontes\\output_rcm'